# Solutions · 04 · Isothermal reactor design

Worked solutions to the exercises of [notebook 04](../notebooks/04_isothermal_reactor_design.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate
from engreact import ideal, kinetics, multiple
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

The packed bed's pellets are replaced by 8 mm ones (halving the Ergun gradient at the same flow... compute it). Recompute alpha and the outlet conversion. If the reaction is diffusion-limited inside the pellets (notebook 11), what else changes?

In [2]:
rho, mu, phi, F_A0, C_A0g, eps, k_prime = 1.15, 1.9e-5, 0.42, 0.02, 40.0, 0.5, 2.0e-5
rate_pbr = lambda X, y: k_prime * kinetics.concentration_from_conversion(C_A0g, X, eps, P_ratio=y)
for Dp in (4e-3, 8e-3):
    beta0 = ideal.ergun(rho * 0.8, rho, mu, Dp, phi)
    alpha = ideal.ergun_alpha(beta0, P0=2e5, A_c=0.015, rho_c=1200.0, phi=phi)
    pb = ideal.pbr_with_pressure_drop(F_A0, 100.0, rate_pbr, alpha, eps)
    print(f"D_p = {Dp*1e3:.0f} mm: Ergun gradient {beta0:.0f} Pa/m, alpha = {alpha:.2e} 1/kg, outlet X = {pb['X'][-1]:.3f}, P/P0 = {pb['y'][-1]:.3f}")
print("larger pellets: less pressure drop, but a larger Thiele modulus - if the reaction is diffusion-limited, k' falls with 1/D_p and conversion drops (notebook 11)")

D_p = 4 mm: Ergun gradient 3168 Pa/m, alpha = 3.03e-03 1/kg, outlet X = 0.933, P/P0 = 0.769
D_p = 8 mm: Ergun gradient 1422 Pa/m, alpha = 1.36e-03 1/kg, outlet X = 0.943, P/P0 = 0.903
larger pellets: less pressure drop, but a larger Thiele modulus - if the reaction is diffusion-limited, k' falls with 1/D_p and conversion drops (notebook 11)


Doubling the pellet diameter roughly halves the inertial Ergun term (and quarters the viscous one), so the
pressure holds up and the conversion rises. The catch is inside the pellet: if internal diffusion limits, the
effective rate constant per kilogram falls in proportion to the diameter, and the larger pellets convert
less - the classic packed-bed trade-off.

## Exercise 2

A membrane reactor removes product B from A <-> B + C (K_c small) as it forms. Model it with `multiple.Network` in a PFR with an extra removal term for B (k_m C_B) and show the conversion exceeding the equilibrium value.

In [3]:
kf, Kc = 0.5, 0.2                                    # A <-> B + C, equilibrium-limited
net = multiple.Network([({"A": -1, "B": 1, "C": 1}, lambda C: kf * (C["A"] - C["B"] * C["C"] / Kc))])
pfr = net.pfr({"A": 1.0}, 40.0)
for k_m in (0.0, 0.5, 2.0):
    mem = multiple.Network([({"A": -1, "B": 1, "C": 1}, lambda C: kf * (C["A"] - C["B"] * C["C"] / Kc)), ({"B": -1}, lambda C, k_m=k_m: k_m * C["B"])])
    out = mem.pfr({"A": 1.0}, 40.0)
    print(f"membrane removal k_m = {k_m}: conversion of A = {1 - out['A'][-1]:.3f}" + ("  (equilibrium limit)" if k_m == 0 else ""))

membrane removal k_m = 0.0: conversion of A = 0.358  (equilibrium limit)
membrane removal k_m = 0.5: conversion of A = 0.974
membrane removal k_m = 2.0: conversion of A = 1.000


Removing B through a membrane keeps the reverse reaction starved, and the conversion climbs past the
equilibrium value - not a violation of thermodynamics but a change of the system: the products never
accumulate. Membrane reactors do this for dehydrogenations (removing hydrogen through palladium).

## Exercise 3

**Implement it yourself:** compute the batch cycle-time optimum: for a fixed dead time of 30 min, scan the conversion at which the batch is stopped and find the one maximising daily production for the saponification.

In [4]:
k2, C_A0, V, dead = 0.11e-3, 200.0, 2.0, 1800.0
Xs = np.linspace(0.3, 0.995, 200)
daily = [C_A0 * V * X * 86400 / (ideal.batch_time_nth_order(k2, C_A0, X, 2) + dead) for X in Xs]
print(f"optimum conversion per batch: {Xs[int(np.argmax(daily))]:.3f} -> {max(daily):.0f} mol/day")

optimum conversion per batch: 0.862 -> 14296 mol/day


The daily output rises with conversion while the reaction is fast and falls when the batch time balloons;
the optimum is where the marginal product per extra minute equals the average - the same argument as for
choosing a harvest time.